# 30 - Cohort biography reader: read the model's memory of one generation

**Author**: kj  **Approach**: read-only life-course narration grounded in the coupled model's own state

Task 70. Every hypothesis round scored *interventions*; this reader instead follows a single **birth cohort**
down its Lexis life-line and tells its story from the numbers the coupled behavioural x Leslie model
(`EmergentModel`) actually produced - nothing invented. `src/sci_demographic_collapse/biography.py` reuses
`run`'s per-calendar-year trajectories and the `ot.CohortMemory` path-integral; it does not re-derive any
dynamics. We read a Korean cohort born into the collapse against a French cohort in the basin, plus one late
Korean cohort whose reproductive window reaches the final simulated year (2124), which is the only case where
the model's aggregate tempo/parity/childlessness/security scalars can be honestly attributed to a cohort.

## GPU selection

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-58ae1f45-295c-681b-60ad-843265f52997"

## Imports

In [2]:
import numpy as np
import torch
from rich import print as rprint
from rich.panel import Panel
from rich.table import Table
from rich.console import Group

from sci_demographic_collapse.emergent import EmergentModel
from sci_demographic_collapse.biography import cohort_biography, SIM_Y0, SIM_YEARS

torch.set_default_dtype(torch.float64)
rprint(f"[green]Device[/green]: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

2026-07-08 12:14:51.188 | INFO     | sci_demographic_collapse.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/sci-demographic-collapse


Device: NVIDIA RTX PRO 4000 Blackwell

## Configuration

The model is instantiated once against the committed UN WPP slice. `COHORTS` are the birth years we narrate:
a matched Korea/France pair born in 2050 (childhood fully inside the 2023-2124 window, so the CohortMemory
childhood integral is defined), and a late Korean cohort (2079) whose reproductive years run to 2124.

In [3]:
from pathlib import Path

PROJ = Path("/home/lab/workspace/learning/projects/sci-demographic-collapse")
m = EmergentModel(data_dir=str(PROJ / "data" / "raw" / "unwpp"))

COHORTS = [("Korea", 2050), ("France", 2050), ("Korea", 2079)]
rprint(f"model span [cyan]{SIM_Y0}[/cyan]-[cyan]{SIM_Y0 + SIM_YEARS - 1}[/cyan]; "
       f"cohorts: {COHORTS}")

model span 2023-2124; cohorts: [('Korea', 2050), ('France', 2050), ('Korea', 2079)]

## The reader

`cohort_biography(model, region, birth_year)` returns a structured record plus a plain-language `narration`.
The Rich renderer below shows the narration in a panel and the per-channel reproductive-window numbers it is
built from in a table - so the prose and its evidence sit side by side. Coupling C, norm N, marriageability q
and TFR are read at cohort resolution; tempo, parity, childlessness and security are the model's end-of-run
aggregates and are only shown when the cohort's window reaches 2124.

In [4]:
def render(rec):
    r = rec
    tab = Table(title=f"reproductive-window channels (ages 27-45, {r['reproductive_window'][0]}-"
                      f"{r['reproductive_window'][1]})", title_style="bold")
    tab.add_column("channel"); tab.add_column("mean", justify="right")
    tab.add_column("first", justify="right"); tab.add_column("last", justify="right")
    tab.add_column("coverage", justify="right")
    labels = {"C": "C coupling", "N": "N norm", "q": "q marriageability", "TFR": "TFR"}
    for ch in ("C", "N", "q", "TFR"):
        s = r["reproductive"][ch]
        if s is None:
            tab.add_row(labels[ch], "-", "-", "-", "out of window"); continue
        tab.add_row(labels[ch], f"{s['mean']:.3f}", f"{s['first']:.3f}", f"{s['last']:.3f}",
                    f"{s['coverage']*100:.0f}%")
    a = r["anchors"]
    anchor_line = (f"[dim]anchors: C0={a['C0']:.2f}  N0={a['NORM0']:.2f}  C_thr={a['C_thr']:.2f}  "
                   f"thN={a['thN']:.2f}  replacement=2.1  ridge=1.5[/dim]")
    body = Group(r["narration"], "", tab, anchor_line)
    colour = "red" if r["region"] == "Korea" else "green"
    rprint(Panel(body, title=f"[bold]{r['region']} - cohort born {r['birth_year']}[/bold]",
                 border_style=colour))


for reg, yr in COHORTS:
    render(cohort_biography(m, reg, yr))

╭─────────────────────────────────────────── Korea - cohort born 2050 ────────────────────────────────────────────╮
│ The Korea cohort born in 2050. Its childhood (ages 0-17) spanned 2050-2067; its reproductive years (ages 27-45) │
│ spanned 2077-2095. The model runs 2023-2124, so only the part of each window inside that span is read from the  │
│ model.                                                                                                          │
│ Childhood partnership climate (the CohortMemory path-integral of coupling relative to today's C0=0.52): -0.134, │
│ i.e. this cohort grew up in a coupling environment below the 2023 baseline (childhood coverage 100%).           │
│ During its reproductive years the coupling channel C averaged 0.282 (from 0.318 to 0.246), below the C_thr=0.66 │
│ coupling-trap ridge and below today's C0=0.52 - weak, trap-side partnership formation. (window coverage 100%)   │
│ The norm climate N averaged 0.420 against the tipping point thN=0.25 and today's N0=0.42 - Korea's reproductive │
│ cohort sat in the trapped childfree-ideal well.                                                                 │
│ Marriageability capital q averaged +0.000 (deviation from 0; intact/lifted relative to the calibrated           │
│ baseline).                                                                                                      │
│ Fertility contributed: the period TFR over its childbearing years averaged 0.320 (from 0.372 to 0.272) -        │
│ sub-replacement and below the 1.5 collapse ridge. This is the PERIOD rate at those calendar years, a proxy for  │
│ the cohort's realised fertility; the model is period-based and does not carry a completed cohort parity per     │
│ birth-cohort.                                                                                                   │
│ Tempo (tau), parity (Pbar), childlessness (rho) and security (S) are exposed by the model only as end-of-run    │
│ (2124) scalars, not per calendar year, so they cannot be attributed to this cohort at cohort resolution - the   │
│ reader deliberately does not invent them.                                                                       │
│                                                                                                                 │
│   reproductive-window channels (ages 27-45, 2077-2095)                                                          │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓                                                        │
│ ┃ channel           ┃  mean ┃ first ┃  last ┃ coverage ┃                                                        │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩                                                        │
│ │ C coupling        │ 0.282 │ 0.318 │ 0.246 │     100% │                                                        │
│ │ N norm            │ 0.420 │ 0.420 │ 0.420 │     100% │                                                        │
│ │ q marriageability │ 0.000 │ 0.000 │ 0.000 │     100% │                                                        │
│ │ TFR               │ 0.320 │ 0.372 │ 0.272 │     100% │                                                        │
│ └───────────────────┴───────┴───────┴───────┴──────────┘                                                        │
│ anchors: C0=0.52  N0=0.42  C_thr=0.66  thN=0.25  replacement=2.1  ridge=1.5                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── France - cohort born 2050 ───────────────────────────────────────────╮
│ The France cohort born in 2050. Its childhood (ages 0-17) spanned 2050-2067; its reproductive years (ages       │
│ 27-45) spanned 2077-2095. The model runs 2023-2124, so only the part of each window inside that span is read    │
│ from the model.                                                                                                 │
│ Childhood partnership climate (the CohortMemory path-integral of coupling relative to today's C0=0.95): -0.043, │
│ i.e. this cohort grew up in a coupling environment below the 2023 baseline (childhood coverage 100%).           │
│ During its reproductive years the coupling channel C averaged 0.869 (from 0.882 to 0.855), above the C_thr=0.66 │
│ coupling-trap ridge and below today's C0=0.95 - partnership formation held above the trap. (window coverage     │
│ 100%)                                                                                                           │
│ The norm climate N averaged 0.140 against the tipping point thN=0.25 and today's N0=0.14 - France's             │
│ reproductive cohort sat in the untrapped well.                                                                  │
│ Marriageability capital q averaged +0.000 (deviation from 0; intact/lifted relative to the calibrated           │
│ baseline).                                                                                                      │
│ Fertility contributed: the period TFR over its childbearing years averaged 1.377 (from 1.414 to 1.341) -        │
│ sub-replacement and below the 1.5 collapse ridge. This is the PERIOD rate at those calendar years, a proxy for  │
│ the cohort's realised fertility; the model is period-based and does not carry a completed cohort parity per     │
│ birth-cohort.                                                                                                   │
│ Tempo (tau), parity (Pbar), childlessness (rho) and security (S) are exposed by the model only as end-of-run    │
│ (2124) scalars, not per calendar year, so they cannot be attributed to this cohort at cohort resolution - the   │
│ reader deliberately does not invent them.                                                                       │
│                                                                                                                 │
│   reproductive-window channels (ages 27-45, 2077-2095)                                                          │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓                                                        │
│ ┃ channel           ┃  mean ┃ first ┃  last ┃ coverage ┃                                                        │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩                                                        │
│ │ C coupling        │ 0.869 │ 0.882 │ 0.855 │     100% │                                                        │
│ │ N norm            │ 0.140 │ 0.140 │ 0.140 │     100% │                                                        │
│ │ q marriageability │ 0.000 │ 0.000 │ 0.000 │     100% │                                                        │
│ │ TFR               │ 1.377 │ 1.414 │ 1.341 │     100% │                                                        │
│ └───────────────────┴───────┴───────┴───────┴──────────┘                                                        │
│ anchors: C0=0.95  N0=0.14  C_thr=0.66  thN=0.25  replacement=2.1  ridge=1.5                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Korea - cohort born 2079 ────────────────────────────────────────────╮
│ The Korea cohort born in 2079. Its childhood (ages 0-17) spanned 2079-2096; its reproductive years (ages 27-45) │
│ spanned 2106-2124. The model runs 2023-2124, so only the part of each window inside that span is read from the  │
│ model.                                                                                                          │
│ Childhood partnership climate (the CohortMemory path-integral of coupling relative to today's C0=0.52): -0.244, │
│ i.e. this cohort grew up in a coupling environment below the 2023 baseline (childhood coverage 100%).           │
│ During its reproductive years the coupling channel C averaged 0.175 (from 0.203 to 0.154), below the C_thr=0.66 │
│ coupling-trap ridge and below today's C0=0.52 - weak, trap-side partnership formation. (window coverage 100%)   │
│ The norm climate N averaged 0.420 against the tipping point thN=0.25 and today's N0=0.42 - Korea's reproductive │
│ cohort sat in the trapped childfree-ideal well.                                                                 │
│ Marriageability capital q averaged +0.000 (deviation from 0; intact/lifted relative to the calibrated           │
│ baseline).                                                                                                      │
│ Fertility contributed: the period TFR over its childbearing years averaged 0.189 (from 0.219 to 0.168) -        │
│ sub-replacement and below the 1.5 collapse ridge. This is the PERIOD rate at those calendar years, a proxy for  │
│ the cohort's realised fertility; the model is period-based and does not carry a completed cohort parity per     │
│ birth-cohort.                                                                                                   │
│ Because this cohort's reproductive window reaches the final simulated year, its end-of-run aggregate state is   │
│ meaningful: mean age at first birth tau=35.3, parity Pbar=1.56, childlessness rho=0.145, security S=0.05.       │
│                                                                                                                 │
│   reproductive-window channels (ages 27-45, 2106-2124)                                                          │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓                                                        │
│ ┃ channel           ┃  mean ┃ first ┃  last ┃ coverage ┃                                                        │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩                                                        │
│ │ C coupling        │ 0.175 │ 0.203 │ 0.154 │     100% │                                                        │
│ │ N norm            │ 0.420 │ 0.420 │ 0.420 │     100% │                                                        │
│ │ q marriageability │ 0.000 │ 0.000 │ 0.000 │     100% │                                                        │
│ │ TFR               │ 0.189 │ 0.219 │ 0.168 │     100% │                                                        │
│ └───────────────────┴───────┴───────┴───────┴──────────┘                                                        │
│ anchors: C0=0.52  N0=0.42  C_thr=0.66  thN=0.25  replacement=2.1  ridge=1.5                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## What the model exposes at cohort resolution - and what it does not

The reader is deliberately honest about the model's memory. At **cohort resolution** (sliceable onto a birth
cohort's calendar window) the model retains four year-by-year signals: coupling **C**, the childfree-norm
share **N**, marriageability capital **q**, and **TFR**. Everything else - tempo tau, parity Pbar,
childlessness rho, security S - `run` returns only as **end-of-run (2124) scalars**, so they cannot be pinned
to a cohort unless its reproductive window reaches the final year. The Korea/France contrast is stark and
entirely model-grounded: the 2050 Korean cohort spends its childbearing years with coupling far **below** the
C_thr=0.66 trap ridge and the norm **above** the thN=0.25 tipping point (trapped childfree well), contributing
a period TFR near 0.3; the French cohort sits **above** the trap with an untrapped norm and a TFR near 1.4.
One faithfulness caveat carried in every narration: the TFR read over a cohort's window is the **period** rate
at those calendar years, a proxy for realised fertility - the model is period-based and does not carry a
completed cohort parity per birth cohort.